In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy as stats
import seaborn as sns
import numpy as np
import platform

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

# 출력 짤림 방지
pd.set_option('display.max_columns', None)

In [ ]:
# 데이터 불러오기
df = pd.read_csv('Courses.csv', parse_dates=['start_time_DI', 'last_event_DI'])
df.head()

In [ ]:
# 퍼널 단계 컬럼(step): 각 학생 별 진행 단계
df['step'] = np.select(
    [
        df['certified'] ==1,
        df['explored'] == 1,
        df['viewed'] == 1,
        df['registered'] == 1,
    ],
    [
        'c',
        'e',
        'v',
        'r'
    ],
    default='None'
)

df.loc[
    (df['step']=='r') & (df['nplay_video'].isna()),
    'nplay_video'
]=0

# 학습 기간 (duration) 컬럼 생성 (같은 날일 경우 1)
df['duration'] = ((df['last_event_DI'] - df['start_time_DI'])).dt.days.astype(int, errors='ignore') + 1

In [ ]:
# 행제거
print('행 제거 작업 시작 전:')
print(df.shape)

# 퍼널 논리적 오류 행 제거
funnel_mask1 = (df['viewed'] == 0) & (df['explored'] == 1)
funnel_mask2 = (df['explored'] == 0) & (df['certified'] == 1)
df = df[~funnel_mask1]
df = df[~funnel_mask2]

# durration 음수 행 제거
duration_mask = df['duration'] <= 0
df = df[~duration_mask]

# # age 13세 미만 행 제거
# age_mask = df['age'] < 13
# df = df[~age_mask]

# 상시 개방된 강의 제거 
course_mask = (df['course_id'] =='HarvardX/CS50x/2012') | (df['course_id'] =='HarvardX/ER22x/2013_Spring') | (df['course_id'] =='HarvardX/CB22x/2013_Spring')
df = df[~course_mask]

# incomplete_flag == 1 제외
df = df[df['incomplete_flag'].isna()]

# 논리적 오류 drop 
rchap_mask = (df['step']=='r') & ((df['nchapters'] > 0) | (df['nplay_video']>0))
df = df[~rchap_mask]

print('행 제거 작업 후:')
print(df.shape)

In [ ]:
# 수치형 컬럼만 선택
numeric_df = df.select_dtypes(include=['float64', 'int64'])

# 상관계수 행렬 계산
corr_matrix = numeric_df.corr()

# 히트맵 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='RdBu_r', center=0)
plt.title('Correlation Matrix of Numerical Variables')
plt.show()

In [ ]:
from scipy import stats

v = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 1) &
    (df['explored']==0) &
    (df['certified']==0) &
    (df['nplay_video'].notna()),
    'nplay_video']

print(v.shape)

r = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 0) &
    (df['explored']==0) &
    (df['certified']==0) &
    (df['nplay_video'].notna()),
    'nplay_video']

print(r.shape)

e = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 1) &
    (df['explored']==1) &
    (df['certified']==0) &
    (df['nplay_video'].notna()),
    'nplay_video']

print(e.shape)

c = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 1) &
    (df['explored']==1) &
    (df['certified']==1) &
    (df['nplay_video'].notna()),
    'nplay_video']

print(c.shape)

In [ ]:
## qqplot

plt.figure(figsize=(8, 6))
stats.probplot(r, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()

plt.figure(figsize=(8, 6))
stats.probplot(v, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()

plt.figure(figsize=(8, 6))
stats.probplot(e, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()

plt.figure(figsize=(8, 6))
stats.probplot(c, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()

In [ ]:
def fff(df):
    sns.histplot(df, kde=True)
    plt.show()

fff(r)
fff(v)
fff(e)
fff(c)

In [ ]:
# 정규성 검정
def ks_test(group):
    sample_mean = np.mean(group)
    s = np.std(group, ddof=1)

    statsa, ps = stats.kstest(group,'norm', args=(sample_mean, s))
    return ps

p_r = ks_test(r)
p_v = ks_test(v)
p_e = ks_test(e)
p_c = ks_test(c)
print(p_r)
print(p_v)
print(p_e)
print(p_c)

In [ ]:
stat_rv, p_rv = stats.mannwhitneyu(r, v, alternative='two-sided')
print(stat_rv, p_rv)
stat_ve, p_ve = stats.mannwhitneyu(v, e, alternative='two-sided')
print(stat_ve, p_ve)
stat_ec, p_ec = stats.mannwhitneyu(e, c, alternative='two-sided')
print(stat_ec, p_ec)

In [ ]:
def rbc(x, y, stat):
    nx = len(x)
    ny = len(y)
    return ((2 * stat) / (nx * ny))-1

print(rbc(r,v,stat_rv))
print(rbc(v,e,stat_ve))
print(rbc(e,c,stat_ec))